[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/14_model_evaluation/01_confusion_matrix_and_metrics.ipynb)

# Bài 01 — Confusion matrix và các metric phân loại

**🎯 Mục tiêu**
- Đọc **confusion matrix**: TP, FP, FN, TN — và vì sao **accuracy** có thể đánh lừa.
- Precision, recall (sensitivity), specificity, F1, $F_\beta$ — cài đặt from scratch và đối chiếu scikit-learn.
- Nhiều lớp: **macro / micro / weighted** averaging; balanced accuracy; **MCC**.
- Biết metric nào trả lời câu hỏi nghiệp vụ nào.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/14_model_evaluation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics as skm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from ml_course import metrics as mm
from ml_course.datasets import make_fraud_transactions
from ml_course.visualization import plot_confusion_matrix, set_style

set_style()

## 1. Confusion matrix

| | Dự đoán **Positive** | Dự đoán **Negative** |
|---|---|---|
| Thật **Positive** | **TP** (bắt đúng) | **FN** (bỏ sót — lỗi loại II) |
| Thật **Negative** | **FP** (báo nhầm — lỗi loại I) | **TN** |

Ví dụ: phát hiện giao dịch gian lận (positive = gian lận, ~2%).

In [ ]:
fraud = make_fraud_transactions(n=20000, fraud_rate=0.02, seed=1)
features = ["amount_vnd", "hour", "distance_from_home_km", "is_foreign", "n_tx_last_24h", "card_age_days"]
X = fraud[features].astype(float).assign(amount_vnd=lambda d: np.log(d.amount_vnd))
y = fraud["is_fraud"].to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
model = LogisticRegression(max_iter=2000).fit(X_tr, y_tr)
proba = model.predict_proba(X_te)[:, 1]
pred = (proba >= 0.5).astype(int)
cm = mm.confusion_matrix(y_te, pred)
print(f"tỉ lệ gian lận trong test: {y_te.mean():.2%}")
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
plot_confusion_matrix(cm, labels=["bình thường", "gian lận"], ax=axes[0], title="Số lượng")
plot_confusion_matrix(cm, labels=["bình thường", "gian lận"], ax=axes[1], normalize=True, title="Chuẩn hoá theo hàng (thật)")
plt.tight_layout()
plt.show()

## 2. Accuracy paradox

In [ ]:
print(f"Logistic Regression (ngưỡng 0.5): accuracy = {mm.accuracy_score(y_te, pred):.4f}")
print(f"'Luôn đoán bình thường'          : accuracy = {mm.accuracy_score(y_te, np.zeros_like(y_te)):.4f}")
print(f"Model bắt được {cm[1, 1]} / {cm[1].sum()} giao dịch gian lận")

Model gần như **không hơn** baseline vô dụng về accuracy — và bỏ sót phần lớn gian lận. Với dữ liệu mất cân bằng, accuracy bị lớp đa số chi phối → cần metric tập trung vào lớp positive.

## 3. Precision, recall, specificity, F1

$$\text{Precision} = \frac{TP}{TP + FP}\ \ \text{(trong các cảnh báo, bao nhiêu đúng?)} \qquad \text{Recall} = \frac{TP}{TP + FN}\ \ \text{(bắt được bao nhiêu % gian lận?)}$$

$$\text{Specificity} = \frac{TN}{TN + FP} \qquad F_1 = \frac{2\cdot P\cdot R}{P + R} \qquad F_\beta = \frac{(1 + \beta^2)\,P\,R}{\beta^2 P + R}$$

- $F_1$: trung bình **điều hoà** — thấp nếu một trong hai thấp.
- $F_\beta$: $\beta > 1$ ưu tiên recall (y tế, gian lận), $\beta < 1$ ưu tiên precision (spam).

In [ ]:
def precision_recall_f1(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


rows = []
for t in [0.5, 0.2, 0.1, 0.05]:
    p_t = (proba >= t).astype(int)
    P, R, F = precision_recall_f1(y_te, p_t)
    rows.append({"ngưỡng": t, "precision": P, "recall": R, "F1": F, "F2 (ưu tiên recall)": skm.fbeta_score(y_te, p_t, beta=2),
                 "specificity": mm.specificity_score(y_te, p_t), "số cảnh báo": int(p_t.sum())})
pd.DataFrame(rows).round(3)

In [ ]:
P, R, F = precision_recall_f1(y_te, (proba >= 0.1).astype(int))
print("khớp sklearn:", np.allclose([P, R, F], [skm.precision_score(y_te, proba >= 0.1), skm.recall_score(y_te, proba >= 0.1),
                                                skm.f1_score(y_te, proba >= 0.1)]))

Hạ ngưỡng → recall tăng, precision giảm: đánh đổi **luôn tồn tại**. Ngưỡng tốt phụ thuộc chi phí của FP và FN (bài 02, 04) — không phải 0.5.

## 4. Nhiều lớp: macro, micro, weighted

Với $K$ lớp, tính precision/recall/F1 cho **từng lớp** (lớp đó là positive, còn lại negative), rồi gộp:
- **Macro**: trung bình đều các lớp — mỗi lớp quan trọng như nhau (lớp hiếm có tiếng nói).
- **Weighted**: trung bình theo số mẫu (support) của lớp.
- **Micro**: cộng TP/FP/FN của mọi lớp rồi tính — với single-label, micro-F1 = accuracy.

In [ ]:
from sklearn.metrics import classification_report

rng = np.random.default_rng(0)
y_true3 = rng.choice(["mèo", "chó", "chim"], 1000, p=[0.7, 0.25, 0.05])
y_pred3 = y_true3.copy()
flip = rng.random(1000) < 0.15
y_pred3[flip] = rng.choice(["mèo", "chó", "chim"], flip.sum(), p=[0.6, 0.3, 0.1])
y_pred3[(y_true3 == "chim") & (rng.random(1000) < 0.6)] = "mèo"            # lớp hiếm bị nhầm nhiều
print(classification_report(y_true3, y_pred3, digits=3))
for avg in ["macro", "weighted", "micro"]:
    print(f"F1 {avg:8s} = {skm.f1_score(y_true3, y_pred3, average=avg):.3f}")
print(f"accuracy    = {skm.accuracy_score(y_true3, y_pred3):.3f}  (= micro-F1)")

Macro-F1 thấp hơn hẳn vì lớp hiếm "chim" kém — weighted/micro che giấu điều đó.

## 5. Balanced accuracy và MCC

- **Balanced accuracy** = trung bình recall các lớp — baseline đoán một lớp luôn cho 1/K.
- **Matthews Correlation Coefficient** (MCC) — tương quan giữa nhãn thật và dự đoán, $\in [-1, 1]$, dùng cả 4 ô của confusion matrix; ổn định với mất cân bằng:

$$\text{MCC} = \frac{TP\cdot TN - FP\cdot FN}{\sqrt{(TP + FP)(TP + FN)(TN + FP)(TN + FN)}}$$

In [ ]:
for name, p_ in [("luôn đoán 0", np.zeros_like(y_te)), ("LR ngưỡng 0.5", pred), ("LR ngưỡng 0.1", (proba >= 0.1).astype(int))]:
    print(f"{name:14s}: accuracy = {skm.accuracy_score(y_te, p_):.3f} | balanced acc = {skm.balanced_accuracy_score(y_te, p_):.3f} | "
          f"MCC = {skm.matthews_corrcoef(y_te, p_):.3f} | F1 = {skm.f1_score(y_te, p_, zero_division=0):.3f}")

## 6. Metric nào cho câu hỏi nào?

| Câu hỏi nghiệp vụ | Metric |
|---|---|
| "Trong các ca model cảnh báo, bao nhiêu % là thật?" (chi phí xử lý cảnh báo) | Precision |
| "Bỏ sót bao nhiêu % ca bệnh / gian lận?" | Recall (1 − recall = tỉ lệ bỏ sót) |
| "Người khoẻ có bị báo nhầm nhiều không?" | Specificity, FPR = 1 − specificity |
| Cân bằng precision–recall, một con số | F1 / $F_\beta$ |
| So sánh model trên dữ liệu mất cân bằng, một con số | MCC, balanced accuracy, PR-AUC (bài 02) |
| Mọi lớp quan trọng như nhau | Macro-F1 |

## 🧠 Kiểm tra nhanh

1. TP = 40, FP = 10, FN = 60, TN = 890. Tính accuracy, precision, recall, F1.
2. Vì sao F1 dùng trung bình điều hoà thay vì trung bình cộng?
3. Bài toán nhiều lớp với một lớp rất hiếm quan trọng — macro hay weighted F1?
4. Model A: precision 0.9, recall 0.3. Model B: precision 0.5, recall 0.8. Sàng lọc ung thư nên chọn model nào? Lọc spam?

## 📌 Tóm tắt

- Confusion matrix là gốc của mọi metric phân loại; accuracy dễ đánh lừa khi mất cân bằng.
- Precision ↔ recall đánh đổi theo ngưỡng; F1/$F_\beta$ gộp hai thứ.
- Nhiều lớp: macro (mỗi lớp như nhau) vs weighted/micro; MCC và balanced accuracy bền với mất cân bằng.

## 📚 Tài liệu tham khảo

- Vũ Hữu Tiệp — [Đánh giá hệ thống phân lớp](https://machinelearningcoban.com/2017/08/31/evaluation/)
- Chicco & Jurman (2020) — *The advantages of the Matthews correlation coefficient (MCC) over F1 score and accuracy*
- scikit-learn User Guide — Metrics and scoring